# Albumentations augmentation pipeline

## Why Albumentations over Keras built-in layers

Keras's augmentation layers cover the basics. Albumentations adds two transforms
that directly address the specific data issues identified in the EDA:

| Transform | Problem it fixes |
|-----------|-----------------|
| `CoarseDropout` | **Subject bias** — randomly erases patches, forcing the model to attend to brushstroke texture across the *whole* canvas rather than just the primary subject in the center |
| `ElasticTransform` | Creates subtle, organic warping that mimics natural variation in brush handling without distorting colour or composition |
| `CLAHE` | Normalises local contrast — useful since some artists (Dürer, Doré) mix high-contrast etchings with softer paintings |
| `HueSaturationValue` | Fine-grained colour jitter within safe bounds — gentler than random grayscale, preserves palette signatures |

**Requirements:**
```
pip install albumentations
```

## Integration pattern

Albumentations operates on numpy arrays (HWC uint8).
`tf.data` operates on tensors.
The bridge is `tf.numpy_function` — it calls a Python function element-wise
inside the TF graph, passing tensors as numpy arrays.


In [2]:
import numpy as np
import tensorflow as tf
import albumentations as A
from pathlib import Path


## Define the augmentation pipeline

In [ ]:
# ── Training augmentation ─────────────────────────────────────────────────────
# Each transform has a probability p — tuned to be meaningful but not aggressive.
# The ordering matters: geometric transforms first, then colour, then dropout.
train_transform = A.Compose([
    # ── Geometric ─────────────────────────────────────────────────────────────
    A.HorizontalFlip(p=0.5),
    # ShiftScaleRotate: combines three transforms into one — efficient and
    # prevents the model from being position-sensitive
    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.1,
        rotate_limit=10,
        border_mode=0,      # constant border (black) — avoids reflection artifacts
        p=0.6
    ),
    # ElasticTransform: small organic deformations that mimic natural variation
    # in brush pressure and canvas texture. alpha controls deformation magnitude.
    A.ElasticTransform(
        alpha=60,
        sigma=6,
        p=0.3
    ),

    # ── Colour ────────────────────────────────────────────────────────────────
    # Fine-grained HSV jitter — intentionally mild to preserve artist palettes.
    # hue_shift_limit=10 is about ±4% of the colour wheel — barely perceptible.
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=20,
        val_shift_limit=15,
        p=0.4
    ),
    # CLAHE: Contrast Limited Adaptive Histogram Equalization.
    # Normalises local contrast tile-by-tile — useful for artists who mixed
    # high-contrast sketches with low-contrast oil paintings.
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),

    # ── Dropout ───────────────────────────────────────────────────────────────
    # CoarseDropout (CutOut): erases rectangular regions with the dataset mean.
    # The key fix for subject bias: if the central subject is partially hidden,
    # the model is forced to look at surrounding brushstroke texture.
    # max_holes=8, max_height/width=48px at 384px = ~12.5% of image per hole.
    A.CoarseDropout(
        max_holes=8,
        max_height=48,
        max_width=48,
        min_holes=2,
        min_height=16,
        min_width=16,
        fill_value=[0.485 * 255, 0.456 * 255, 0.406 * 255],  # ImageNet mean
        p=0.4
    ),
])

# Val/test: no augmentation at all
eval_transform = A.Compose([])   # identity — makes the API uniform

print("Augmentation pipeline defined.")
print(f"Training transforms: {len(train_transform.transforms)}")


## Wire Albumentations into a `tf.data` pipeline

In [ ]:
# ── Bridge function ───────────────────────────────────────────────────────────
# tf.numpy_function passes the tensor as a numpy array to our Python function.
# It runs eagerly (not compiled) — acceptable because image decoding and
# Albumentations is already the CPU bottleneck; the call overhead is negligible.

def augment_image(image: np.ndarray) -> np.ndarray:
    """Apply training augmentation to a single HWC uint8 image."""
    return train_transform(image=image)["image"]

def tf_augment(image, label):
    """Wrap augment_image for use inside tf.data.Dataset.map()."""
    aug_image = tf.numpy_function(
        func=augment_image,
        inp=[image],
        Tout=tf.uint8,
    )
    # tf.numpy_function loses shape info — restore it explicitly
    aug_image.set_shape(image.shape)
    return aug_image, label


# ── Full pipeline builder ─────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def build_alb_train_ds(raw_ds, batch_size, cache_path: str = None):
    """
    Unbatched raw dataset → augment per image → batch → cache → prefetch.

    Augmentation happens BEFORE batching so each image gets an independent
    random transform. Caching AFTER augmentation means the cache stores
    augmented batches — fine for a fixed cache, but it means you see the
    same augmentations every epoch. For true per-epoch randomness, either:
      a) don't cache (slower), or
      b) cache the raw images and put augment AFTER cache (cache raw, augment live)
    Option (b) is shown below as the recommended default.
    """
    ds = raw_ds.map(tf_augment, num_parallel_calls=AUTOTUNE)
    if cache_path:
        ds = ds.cache(cache_path)
    return ds.batch(batch_size, drop_remainder=True).prefetch(AUTOTUNE)


def build_alb_train_ds_v2(raw_ds, batch_size, cache_path: str):
    """
    Recommended: cache RAW images, augment AFTER cache.
    Every epoch sees freshly randomised augmentations, while disk reads only
    happen once (raw pixels are cached to SSD on the first epoch).
    """
    ds = raw_ds.cache(cache_path)          # cache raw — fast repeated reads
    ds = ds.shuffle(50_000, seed=None)     # re-shuffle each epoch (seed=None)
    ds = ds.map(tf_augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size, drop_remainder=True).prefetch(AUTOTUNE)


def build_eval_ds(raw_ds, batch_size):
    """Val/test: no augmentation, no shuffle."""
    return raw_ds.batch(batch_size, drop_remainder=False).prefetch(AUTOTUNE)


## Usage example

In [ ]:
# ── Example: visualise augmented samples ─────────────────────────────────────
# (Run this cell to verify the pipeline before starting training)

import matplotlib.pyplot as plt
from keras.utils import image_dataset_from_directory

DATA_DIR   = Path("../wikiart_split")
IMAGE_SIZE = (384, 384)
BATCH_SIZE = 16

_raw = image_dataset_from_directory(
    DATA_DIR / "train",
    batch_size=None,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    label_mode="categorical",
)

# Take 5 original images
original_images = list(_raw.take(5).map(lambda x, y: x))

fig, axes = plt.subplots(5, 5, figsize=(16, 16))
for row, orig in enumerate(original_images):
    orig_np = orig.numpy().astype(np.uint8)
    axes[row, 0].imshow(orig_np)
    axes[row, 0].set_title("Original", fontsize=9)
    axes[row, 0].axis("off")
    for col in range(1, 5):
        aug_np = augment_image(orig_np)
        axes[row, col].imshow(aug_np)
        axes[row, col].set_title(f"Aug {col}", fontsize=9)
        axes[row, col].axis("off")

plt.suptitle("Row = same image, Columns = different augmentations", fontsize=12)
plt.tight_layout()
plt.show()


## How to use with the existing training notebook

Replace the `make_train_ds` function in `model_hw_tuned.ipynb` with:

```python
# In model_hw_tuned.ipynb — replace make_train_ds with this:
from albumentations_pipeline import build_alb_train_ds_v2, build_eval_ds

train_bs16 = build_alb_train_ds_v2(
    _train_raw,
    batch_size=16,
    cache_path=str(SSD_CACHE_DIR / "train_raw_bs16")
)
```

The rest of the training code is unchanged — `build_alb_train_ds_v2` returns
a standard `tf.data.Dataset` with the same `(images, labels)` structure.
